# 第 10 章：LoRA / QLoRA 参数高效微调

这个 notebook 对应 `lessons/10_lora_qlora.md`，演示教学版 LoRA adapter：冻结 base linear、注入 target modules、统计可训练参数、训练一步检查 adapter 更新、保存/加载 adapter，以及 QLoRA 配置边界。

In [ ]:
import tempfile
from pathlib import Path

import torch
from torch import nn

from src.finetune.lora import (
    LoRAConfig,
    QLoRAConfig,
    estimate_linear_parameter_count,
    estimate_lora_parameter_count,
    inject_lora_adapters,
    load_lora_adapter,
    qlora_memory_note,
    save_lora_adapter,
    trainable_parameter_summary,
    write_lora_manifest,
)

## 1. LoRA 参数量

全量 Linear 参数是 `out * in`，LoRA 新增参数是 `r * (in + out)`。

In [ ]:
full = estimate_linear_parameter_count(4096, 4096, bias=False)
lora = estimate_lora_parameter_count(4096, 4096, rank=8)
print("full:", full)
print("lora:", lora)
print("ratio:", round(lora / full, 6))

## 2. 注入 Target Modules

真实 PEFT 会按模块名注入 adapter。这里用一个 tiny model 演示 target module 匹配和 base 冻结。

In [ ]:
class TinyTargetModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(4, 6)
        self.v_proj = nn.Linear(4, 6)
        self.other = nn.Linear(6, 2)

    def forward(self, x):
        return self.other(torch.relu(self.q_proj(x) + self.v_proj(x)))


model = TinyTargetModel()
config = LoRAConfig(r=2, alpha=4, target_modules=("q_proj", "v_proj"))
matched = inject_lora_adapters(model, config)
summary = trainable_parameter_summary(model)
print("matched:", matched)
print("trainable ratio:", round(summary.ratio, 4))
print("q_proj base trainable:", model.q_proj.base.weight.requires_grad)

## 3. 训练一步：base 不变，adapter 变化

LoRA 的关键验收是 base frozen weight 不更新，而 adapter 权重更新。

In [ ]:
before_base = model.q_proj.base.weight.detach().clone()
before_adapter = model.q_proj.lora_B.weight.detach().clone()
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=0.1)
x = torch.randn(5, 4)
target = torch.randn(5, 2)

loss = nn.functional.mse_loss(model(x), target)
optimizer.zero_grad()
loss.backward()
optimizer.step()

base_changed = not torch.allclose(before_base, model.q_proj.base.weight.detach())
adapter_changed = not torch.allclose(before_adapter, model.q_proj.lora_B.weight.detach())
print("base changed:", base_changed)
print("adapter changed:", adapter_changed)

## 4. 保存 Adapter 与 Manifest

Adapter 要记录 base model id/revision、rank、alpha、target modules 和可训练参数比例。

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    adapter_path = Path(tmpdir) / "adapter.pt"
    manifest_path = Path(tmpdir) / "lora_manifest.json"
    save_lora_adapter(adapter_path, model, config, extra={"step": 1})
    write_lora_manifest(manifest_path, config, summary)
    fresh = TinyTargetModel()
    loaded_config, extra = load_lora_adapter(adapter_path, fresh)
    print(loaded_config)
    print(extra)
    print(manifest_path.read_text())

## 5. QLoRA 边界

QLoRA 的重点是 4-bit frozen base + trainable LoRA adapter，但激活、KV cache、batch 和序列长度仍然会消耗显存。

In [ ]:
print(qlora_memory_note(QLoRAConfig()))